In [1]:
import sys
sys.path.insert(0, '..')
from dependencies import *

subject_model_predictors = eelbrain.load.unpickle(PREDICTOR_DIR / f'~concatenated_predictors.pickle')
models = get_models()

In [2]:
# ------------------------------------------------
# Directories
# ------------------------------------------------
# Decoder evaluation (statistics only)
# ------------------------------------------------
r_values = {}
r2_values = {}
rmse_values = {model: [] for model in models}

for model in models:
    r_values[model] = []
    r2_values[model] = []

for subject in SUBJECTS:

    print(f'\n===== Subject {subject} =====')

    eeg = eelbrain.load.unpickle(STIMULUS_DIR / f'{subject}concatenated_eeg.pickle')

    for model in models:

        print(f'\nModel: {model}')

        # Load decoder TRF
        trf_decoder = eelbrain.load.unpickle(
            TRF_DIR / subject / f'{subject} decoder-{model}.pickle'
        )

        predictors = subject_model_predictors[subject][model]

        # Predict stimulus from EEG
        predicted = eelbrain.convolve(trf_decoder.h_scaled, eeg)

        # Build actual stimulus (sum if multiple)
        stimulus = sum(predictors) if len(predictors) > 1 else predictors[0]

        # Convert to numpy arrays
        env = stimulus.x
        pred = predicted.x

        # Correlation and r²
        r = np.corrcoef(env, pred)[0,1]
        r2 = r**2

        r_values[model].append(r)
        r2_values[model].append(r2)

        # ----------------------------------------
        # RMSE (normalized ONLY here)
        # ----------------------------------------
        env_norm = (env - np.mean(env)) / np.std(env)
        pred_norm = (pred - np.mean(pred)) / np.std(pred)

        rmse = np.sqrt(np.mean((env_norm - pred_norm)**2))

        rmse_values[model].append(rmse)

        print(f'Normalized RMSE = {rmse:.4f}')

        print(f'r = {r:.4f}, r² = {r2:.4f}, RMSE = {rmse:.4f}')



===== Subject S05 =====

Model: envelope_log
Normalized RMSE = 1.2933
r = 0.1637, r² = 0.0268, RMSE = 1.2933

Model: envelope_onset
Normalized RMSE = 1.3368
r = 0.1064, r² = 0.0113, RMSE = 1.3368

Model: envelope_log_onset
Normalized RMSE = 1.2933
r = 0.1637, r² = 0.0268, RMSE = 1.2933

===== Subject S34 =====

Model: envelope_log
Normalized RMSE = 1.2437
r = 0.2266, r² = 0.0514, RMSE = 1.2437

Model: envelope_onset
Normalized RMSE = 1.3210
r = 0.1275, r² = 0.0163, RMSE = 1.3210

Model: envelope_log_onset
Normalized RMSE = 1.2437
r = 0.2266, r² = 0.0514, RMSE = 1.2437

===== Subject S35 =====

Model: envelope_log
Normalized RMSE = 1.2451
r = 0.2249, r² = 0.0506, RMSE = 1.2451

Model: envelope_onset
Normalized RMSE = 1.3564
r = 0.0801, r² = 0.0064, RMSE = 1.3564

Model: envelope_log_onset
Normalized RMSE = 1.2451
r = 0.2249, r² = 0.0506, RMSE = 1.2451

===== Subject S03 =====

Model: envelope_log
Normalized RMSE = 1.2408
r = 0.2302, r² = 0.0530, RMSE = 1.2408

Model: envelope_onset
Nor

In [3]:
for model in r_values:
    mean_r = np.mean(r_values[model])
    std_r = np.std(r_values[model])
    mean_r2 = np.mean(r2_values[model])
    std_r2 = np.std(r2_values[model])
    print(f'{model}: r = {mean_r:.4f} ± {std_r:.4f}, r² = {mean_r2:.4f} ± {std_r2:.4f}')

envelope_log: r = 0.2094 ± 0.0830, r² = 0.0507 ± 0.0302
envelope_onset: r = 0.0850 ± 0.0361, r² = 0.0085 ± 0.0051
envelope_log_onset: r = 0.2094 ± 0.0830, r² = 0.0507 ± 0.0302


In [4]:
t_stat, p_val = ttest_rel(r_values['envelope_log'], r_values['envelope_onset'])
print(f'Envelope log vs. onset: t={t_stat:.2f}, p={p_val:.4f}')

Envelope log vs. onset: t=13.08, p=0.0000


In [ ]:
# ------------------------------------------------
# Storage
# ------------------------------------------------
r_values_universal = {model: [] for model in models}
r2_values_universal = {model: [] for model in models}
rmse_values = {model: [] for model in models}

# ------------------------------------------------
# Loop over models
# ------------------------------------------------
for model in models:

    print(f'\n===== Universal decoder: {model} =====')

    trf_universal = eelbrain.load.unpickle(
        TRF_DIR / f'decoder-universal-trf-{model}.pickle'
    )

    for subject in SUBJECTS:

        print(f'\nSubject: {subject}')

        eeg = eelbrain.load.unpickle(
            EEG_DIR / subject / f'{subject}concatenated_eeg.pickle'
        )

        predictors = subject_model_predictors[subject][model]

        predicted = eelbrain.convolve(trf_universal, eeg)

        stimulus = predictors[0]

        env = stimulus.x
        pred = predicted.x

        # ----------------------------------------
        # Correlation
        # ----------------------------------------
        r = np.corrcoef(env, pred)[0,1]
        r2 = r**2

        r_values_universal[model].append(r)
        r2_values_universal[model].append(r2)

        print(f'r = {r:.4f}, r² = {r2:.4f}')

        # ----------------------------------------
        # RMSE (normalized ONLY here)
        # ----------------------------------------
        env_norm = (env - np.mean(env)) / np.std(env)
        pred_norm = (pred - np.mean(pred)) / np.std(pred)

        rmse = np.sqrt(np.mean((env_norm - pred_norm)**2))

        rmse_values[model].append(rmse)

        print(f'Normalized RMSE = {rmse:.4f}')
    

print('\n===== Universal decoder summary =====')

for model in models:
    mean_r = np.mean(r_values_universal[model])
    std_r = np.std(r_values_universal[model])
    mean_r2 = np.mean(r2_values_universal[model])
    std_r2 = np.std(r2_values_universal[model])
    mean_rmse = np.mean(rmse_values[model])
    std_rmse = np.std(rmse_values[model])

    print(f'{model}: r = {mean_r:.4f} ± {std_r:.4f}, '
          f'r² = {mean_r2:.4f} ± {std_r2:.4f}, '
          f'RMSE = {mean_rmse:.4f} ± {std_rmse:.4f}')



===== Universal decoder: envelope_log =====

Subject: S05
r = 0.0476, r² = 0.0023
Normalized RMSE = 1.3802

Subject: S34
r = 0.1022, r² = 0.0105
Normalized RMSE = 1.3400

Subject: S35
r = 0.0998, r² = 0.0100
Normalized RMSE = 1.3418

Subject: S03
r = 0.1124, r² = 0.0126
Normalized RMSE = 1.3324

Subject: S04
r = 0.1438, r² = 0.0207
Normalized RMSE = 1.3086

Subject: S44
r = 0.0510, r² = 0.0026
Normalized RMSE = 1.3777

Subject: S26
r = 0.0015, r² = 0.0000
Normalized RMSE = 1.4131

Subject: S19
r = 0.1327, r² = 0.0176
Normalized RMSE = 1.3170

Subject: S21
r = 0.1425, r² = 0.0203
Normalized RMSE = 1.3096

Subject: S17
r = 0.1345, r² = 0.0181
Normalized RMSE = 1.3157

Subject: S10
r = 0.0633, r² = 0.0040
Normalized RMSE = 1.3687

Subject: S42
r = 0.0687, r² = 0.0047
Normalized RMSE = 1.3648

Subject: S45
r = 0.0160, r² = 0.0003
Normalized RMSE = 1.4029

Subject: S11
r = 0.0991, r² = 0.0098
Normalized RMSE = 1.3423

Subject: S16
r = 0.1953, r² = 0.0381
Normalized RMSE = 1.2686

Subject: 